# 04 · Fichas de especie

Reúne lo que el prototipo necesita mostrar junto a una predicción: nombre común,
resumen y una imagen de referencia del propio corpus.

Las imágenes son obra de terceros bajo licencias Creative Commons, así que cada
ficha guarda su atribución y la ficha no sirve sin mostrarla.

In [14]:
import json
import shutil
import time

import requests

from corpus import (AUDITORIA, FICHAS, FICHAS_JSON, LISTA_ESPECIES, METADATA,
                    PROCESADO, leerCsv)

API = "https://api.inaturalist.org/v1"
PAUSA_API = 1.5
TANDA = 30          # tope de ids por llamada a /taxa

especies = json.loads(LISTA_ESPECIES.read_text())["species"]
porTaxon = {e["taxon_id"]: e for e in especies}
print(f"{len(especies)} especies")

100 especies


In [15]:
def pedir(sesion, ruta, params=None, intentos=5):
    """Reintenta con espera creciente ante 429 y cortes de red."""
    for intento in range(intentos):
        try:
            respuesta = sesion.get(f"{API}/{ruta}", params=params or {}, timeout=30)
        except requests.RequestException as e:
            print(f"    {type(e).__name__}, reintento en {2 ** intento}s")
            time.sleep(2 ** intento)
            continue
        if respuesta.status_code == 429:
            espera = float(respuesta.headers.get("Retry-After", 2 ** intento))
            print(f"    429, esperando {espera:.0f}s")
            time.sleep(espera)
            continue
        respuesta.raise_for_status()
        return respuesta.json()
    raise RuntimeError(f"{ruta} falló tras {intentos} intentos")

## 1 · Nombre común y resumen

In [16]:
sesion = requests.Session()
sesion.headers["User-Agent"] = "butterfly-dataset/1.0"

taxones = {}
ids = sorted(porTaxon)
for comienzo in range(0, len(ids), TANDA):
    lote = ids[comienzo:comienzo + TANDA]
    for t in pedir(sesion, "taxa/" + ",".join(str(i) for i in lote))["results"]:
        taxones[t["id"]] = t
    time.sleep(PAUSA_API)

sinComun = [porTaxon[i]["name"] for i in ids
            if not taxones.get(i, {}).get("preferred_common_name")]
sinResumen = [porTaxon[i]["name"] for i in ids
              if not taxones.get(i, {}).get("wikipedia_summary")]
print(f"{len(taxones)}/{len(ids)} taxones resueltos")
print(f"sin nombre común: {len(sinComun)} · sin resumen: {len(sinResumen)}")

    ConnectionError, reintento en 1s
    ConnectionError, reintento en 1s
100/100 taxones resueltos
sin nombre común: 7 · sin resumen: 28


### Respaldo de género y familia

In [17]:
def resumenDeRango(nombre, rango):
    hallados = pedir(sesion, "taxa", {"q": nombre, "rank": rango, "per_page": 1})["results"]
    time.sleep(PAUSA_API)
    if not hallados:
        return "", ""
    taxon = pedir(sesion, f"taxa/{hallados[0]['id']}")["results"][0]
    time.sleep(PAUSA_API)
    return taxon.get("wikipedia_summary") or "", taxon.get("wikipedia_url") or ""


respaldos = {}
for taxonId in ids:
    if taxones[taxonId].get("wikipedia_summary"):
        continue
    entrada = porTaxon[taxonId]
    for nivel, nombre, rango in (("genero", entrada["name"].split()[0], "genus"),
                                 ("familia", entrada["familia"], "family")):
        resumen, url = resumenDeRango(nombre, rango)
        if resumen:
            respaldos[taxonId] = (nivel, nombre, resumen, url)
            break

print(f"{len(respaldos)}/{len(sinResumen)} resueltos por género o familia")

28/28 resueltos por género o familia


## 2 · Imagen de referencia

La más nítida de cada especie entre las aceptadas del split de entrenamiento,
descartando antes los recortes que la ampliación o el encuadre arruinan.

In [18]:
auditoria = [f for f in leerCsv(AUDITORIA)
             if f["veredicto"] == "aceptada" and f["split"] == "train"]
atribucionDe = {m["archivo"]: (m["atribucion"], m["licencia"])
                for m in leerCsv(METADATA)}


def presentable(fila):
    area = ((float(fila["x2"]) - float(fila["x1"]))
            * (float(fila["y2"]) - float(fila["y1"]))
            / (float(fila["ancho"]) * float(fila["alto"])))
    return (float(fila["ladoOriginal"]) >= 448
            and float(fila["confianza"]) >= 0.30
            and 0.25 <= area <= 0.80)


porEspecie = {}
for fila in auditoria:
    if presentable(fila):
        porEspecie.setdefault(fila["especie"], []).append(fila)

mejorPorEspecie = {especie: max(filas, key=lambda f: float(f["nitidez"]))
                   for especie, filas in porEspecie.items()}

FICHAS.mkdir(exist_ok=True)
for especie, fila in mejorPorEspecie.items():
    shutil.copyfile(PROCESADO / especie / fila["archivo"], FICHAS / f"{especie}.jpg")

print(f"{len(mejorPorEspecie)} imágenes en {FICHAS.name}/")

100 imágenes en fichas/


## 3 · Escribir las fichas

In [19]:
fichas = {}
for taxonId, entrada in porTaxon.items():
    especie = entrada["name"].lower().replace(" ", "_")
    taxon = taxones[taxonId]
    fila = mejorPorEspecie[especie]
    atribucion, licencia = atribucionDe[fila["archivo"]]

    nivel, ambito = "especie", entrada["name"]
    resumen = taxon.get("wikipedia_summary") or ""
    wikipedia = taxon.get("wikipedia_url") or ""
    if not resumen:
        nivel, ambito, resumen, wikipedia = respaldos[taxonId]

    fichas[especie] = {
        "cientifico": entrada["name"],
        "comun": taxon.get("preferred_common_name", ""),
        "familia": entrada["familia"],
        "taxon_id": taxonId,
        "resumen": resumen,
        "resumen_nivel": nivel,
        "resumen_ambito": ambito,
        "wikipedia": wikipedia,
        "inaturalist": f"https://www.inaturalist.org/taxa/{taxonId}",
        "imagen": f"fichas/{especie}.jpg",
        "atribucion": atribucion,
        "licencia": licencia,
    }

FICHAS_JSON.write_text(json.dumps(fichas, indent=1, ensure_ascii=False), encoding="utf-8")
print(f"{len(fichas)} fichas en {FICHAS_JSON.name}")
for nivel in ("especie", "genero", "familia"):
    print(f"resumen de {nivel}: {sum(1 for f in fichas.values() if f['resumen_nivel'] == nivel)}")

100 fichas en fichas.json
resumen de especie: 72
resumen de genero: 26
resumen de familia: 2


In [20]:
import pandas as pd

pd.DataFrame(fichas).T[["cientifico", "comun", "familia", "licencia"]].head(12)

,cientifico,comun,familia,licencia
anartia_amathea,Anartia amathea,Red Peacock,Nymphalidae,cc-by-nc
heliconius_doris,Heliconius doris,Doris Longwing,Nymphalidae,cc-by
anartia_jatrophae,Anartia jatrophae,White Peacock,Nymphalidae,cc-by-nc
heraclides_thoas,Heraclides thoas,Thoas Swallowtail,Papilionidae,cc-by-nc
hamadryas_feronia,Hamadryas feronia,Variable Cracker,Nymphalidae,cc-by-nc
siproeta_stelenes,Siproeta stelenes,Malachite,Nymphalidae,cc-by-nc
melanis_electron,Melanis electron,Orange-barred Pixie,Riodinidae,cc-by-nc
rhetus_periander,Rhetus periander,Periander Metalmark,Riodinidae,cc-by
hamadryas_amphinome,Hamadryas amphinome,Red Cracker,Nymphalidae,cc-by-nc
colobura_dirce,Colobura dirce,Dirce Beauty,Nymphalidae,cc-by-nc
